# QLoRA Fine-tuning: Qwen2.5-1.5B on ASAP (大众点评)

**Runtime**: T4 GPU · **Est. time**: 2-3 hours · **Dataset**: 3,200 ASAP samples

Before running: 运行时 → 更改运行时类型 → T4 GPU

In [ ]:
!pip install -q unsloth
!pip install -q trl datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

TRAIN_PATH = '/content/drive/MyDrive/review-intelligence/train.jsonl'
VAL_PATH   = '/content/drive/MyDrive/review-intelligence/val.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/review-intelligence/qwen-asap-qlora'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Drive mounted, output dir ready')

In [ ]:
import torch, json
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

with open(TRAIN_PATH) as f: train_data = [json.loads(l) for l in f]
with open(VAL_PATH) as f:   val_data   = [json.loads(l) for l in f]
print(f'Train: {len(train_data)}, Val: {len(val_data)}')

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none',
    use_gradient_checkpointing='unsloth',
)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

def fmt(s):
    return {'text': tokenizer.apply_chat_template(s['messages'], tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_data).map(fmt)
val_ds   = Dataset.from_list(val_data).map(fmt)
print(f'Formatted. Sample:\n{train_ds[0]["text"][:300]}')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    dataset_text_field='text', max_seq_length=1024,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05, learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=50,
        evaluation_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='eval_loss',
        optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='cosine', seed=42, report_to='none',
    ),
)

stats = trainer.train()
print(f'Done! {stats.metrics["train_runtime"]:.0f}s')

In [ ]:
FINAL = OUTPUT_DIR + '/final'
model.save_pretrained(FINAL)
tokenizer.save_pretrained(FINAL)
print(f'Saved to {FINAL}')
for f in os.listdir(FINAL):
    print(f'  {f}: {os.path.getsize(os.path.join(FINAL,f))/1e6:.0f} MB')

In [ ]:
# Quick test
FastLanguageModel.for_inference(model)
SYSTEM = '你是餐厅经营分析助手。分析中文餐厅评论，输出结构化 JSON。只输出 JSON。'

for review in ['菜非常好吃，服务态度很好！', '等了一小时，服务差，菜也凉了。']:
    msgs = [{'role':'system','content':SYSTEM}, {'role':'user','content':f'评论：{review}'}]
    inp = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(inp, max_new_tokens=256, temperature=0.1, do_sample=True)
    print(f'\n评论: {review}')
    print(f'输出: {tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True)}')

## 完成后
1. 从 Drive 下载 `qwen-asap-qlora/final/` 到本地 `models/qwen-asap-qlora/`
2. 本地跑 `python src/06_evaluate_finetuned.py`
3. 把数字更新到 Streamlit app